# 5. Hyperparameter Tuning

## Purpose
Improve model performance by finding the optimal hyperparameters for the
selected models using Bayesian optimisation (Optuna) or random search.
Tuning is performed using cross validation only — the test set is not
touched at this stage.

## Inputs
- `data/processed/` — final feature selected preprocessed data
- `data/raw/` — raw data reloaded for CatBoost
- `config.yaml` — tuning method, trials, CV folds, models to tune
- `artifacts/baseline_trainer.pkl` — for task type detection and
  baseline score comparison

## Outputs
- `models/tuned/` — all tuned models fitted on full training data
- `artifacts/tuning_results.pkl` — full Optuna study or RandomizedSearchCV
  results for each model
- `artifacts/tuned_models.pkl` — dictionary of fitted tuned models

## Decisions for the user
- Set `tuning.method` in `config.yaml` — optuna (recommended) or
  random_search
- Set `tuning.n_trials` in `config.yaml` — more trials finds better
  parameters but takes longer. 50 is a good starting point, 200+ for
  final optimisation
- Set `tuning.models_to_tune` in `config.yaml` — typically the top 1-2
  models from the baseline leaderboard
- Review best parameters and consider whether ranges in
  `src/models/training/param_spaces.py` need widening for your dataset



In [1]:
import pandas as pd
import numpy as np
import joblib
import copy

from src.config.settings import load_config
from src.config.paths import (
    PROCESSED_DATA_DIR,
    INTERIM_DATA_DIR,
    TUNED_MODELS_DIR,
    ARTIFACTS_DIR
)
from src.data.load_data import load_raw_data
from src.features.pipeline import FeatureEngineeringPipeline
from src.models.training.model_registry import MODEL_REGISTRY
from src.models.training.param_spaces import PARAM_SPACES, RANDOM_SEARCH_SPACES
from src.models.training.train_tuned import run_tuning, get_scoring_metric
from src.models.training.save_load import save_model

c:\Users\kigr\anaconda3\envs\ml-template\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load config

In [2]:
config = load_config()

TARGET = config["target"]
DROP_COLS = config["drop_columns"]
SPLIT_CONFIG = config["split"]
PIPELINE_CONFIG = config["pipeline"]
TUNING_CONFIG = config["tuning"]

print(f"Target:          {TARGET}")
print(f"Tuning method:   {TUNING_CONFIG['method']}")
print(f"Models to tune:  {TUNING_CONFIG['models_to_tune']}")
print(f"Trials:          {TUNING_CONFIG['n_trials']}")

Target:          Survived
Tuning method:   optuna
Models to tune:  ['xgboost_classifier']
Trials:          50


## 2. Load processed data

In [3]:
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (712, 57)
X_test shape:  (179, 57)


## 3. Load raw data for CatBoost

In [4]:
from sklearn.model_selection import train_test_split

df = load_raw_data()
X_raw = df.drop(columns=[TARGET] + DROP_COLS)
y_all = df[TARGET]

stratify_col = y_all if SPLIT_CONFIG["stratify"] else None

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y_all,
    test_size=SPLIT_CONFIG["test_size"],
    random_state=SPLIT_CONFIG["random_state"],
    stratify=stratify_col
)

## 4. Detect task type
Task type is inferred from the baseline trainer to ensure consistency
across all notebooks.

In [5]:
baseline_trainer = joblib.load(ARTIFACTS_DIR / "baseline_trainer.pkl")
task = baseline_trainer.task_

print(f"Task detected: {task}")

scoring = get_scoring_metric(task, config)
print(f"Scoring metric: {scoring}")

Task detected: classification
Scoring metric: roc_auc


## 5. Tune each model
Runs hyperparameter search for each model in `tuning.models_to_tune`.

In [6]:
tuned_results = {}

for model_name in TUNING_CONFIG["models_to_tune"]:

    print(f"\n{'='*50}")
    print(f"Tuning: {model_name}")
    print(f"{'='*50}")

    # Get model class from registry
    if model_name not in MODEL_REGISTRY:
        print(f"Skipping {model_name}: not found in registry.")
        continue

    model_class = MODEL_REGISTRY[model_name]["model"]

    # Get param space
    if model_name not in PARAM_SPACES:
        print(f"Skipping {model_name}: no param space defined.")
        continue

    # CatBoost uses raw data, all others use processed data
    is_catboost = "catboost" in model_name
    X_input = X_train_raw if is_catboost else X_train

    # Get appropriate param space based on tuning method
    method = TUNING_CONFIG["method"]
    if method == "optuna":
        param_space = PARAM_SPACES[model_name]
    else:
        if model_name not in RANDOM_SEARCH_SPACES:
            print(f"Skipping {model_name}: no random search space defined.")
            continue
        param_space = RANDOM_SEARCH_SPACES[model_name]
        base_model = model_class(**MODEL_REGISTRY[model_name]["default_params"])

    # Run tuning
    result = run_tuning(
        model_class=model_class,
        X_train=X_input,
        y_train=y_train.values,
        task=task,
        config=config,
        param_space=param_space,
        base_model=base_model if method == "random_search" else None,
    )

    tuned_results[model_name] = result


Tuning: xgboost_classifier


Tuning method:  optuna

Scoring metric: roc_auc

Trials/iters:   50

CV folds:       5

Tuning with Optuna...

Best score: 0.8835

Best params: {'n_estimators': 374, 'max_depth': 10, 'learning_rate': 0.03508936303680589, 'subsample': 
0.9975092365680626, 'colsample_bytree': 0.7874820875551369, 'min_child_weight': 10, 'gamma': 0.7011625953937599, 
'reg_alpha': 0.03358588493447223, 'reg_lambda': 1.9909662115423685}

## 6. Tuning results summary
Review the improvement over baseline for each model. Review the best parameters to understand what the search found.

In [7]:
print("\nTuning Results Summary:")
print("=" * 50)

for model_name, result in tuned_results.items():
    method = TUNING_CONFIG["method"]

    if method == "optuna":
        best_score = result.best_value
        best_params = result.best_params
    else:
        best_score = result.best_score_
        best_params = result.best_params_

    # Get baseline score for comparison
    baseline_row = baseline_trainer.leaderboard_[
        baseline_trainer.leaderboard_["model_name"] == model_name
    ]
    baseline_score = baseline_row["primary_metric_value"].values[0] \
        if len(baseline_row) > 0 else None

    print(f"\nModel: {model_name}")
    if baseline_score is not None:
        print(f"Baseline CV {scoring}: {baseline_score:.4f}")
    print(f"Tuned CV {scoring}:    {best_score:.4f}")
    if baseline_score is not None:
        print(f"Improvement:          {best_score - baseline_score:+.4f}")
    print(f"Best params: {best_params}")


Tuning Results Summary:

Model: xgboost_classifier
Baseline CV roc_auc: 0.8829
Tuned CV roc_auc:    0.8835
Improvement:          +0.0006
Best params: {'n_estimators': 374, 'max_depth': 10, 'learning_rate': 0.03508936303680589, 'subsample': 0.9975092365680626, 'colsample_bytree': 0.7874820875551369, 'min_child_weight': 10, 'gamma': 0.7011625953937599, 'reg_alpha': 0.03358588493447223, 'reg_lambda': 1.9909662115423685}


## 7. Fit final tuned models
The best parameters from the search are used to fit a final model on
the full training data. This is the model that will be evaluated on
the test set in notebook 7.

In [8]:
print("\nFitting final tuned models on full training data...")

tuned_models = {}

for model_name, result in tuned_results.items():
    method = TUNING_CONFIG["method"]
    model_class = MODEL_REGISTRY[model_name]["model"]
    is_catboost = "catboost" in model_name

    if method == "optuna":
        best_params = result.best_params
    else:
        best_params = result.best_params_

    if is_catboost:
        # CatBoost — fit on raw data
        from src.models.training.trainer import ModelTrainer
        trainer = ModelTrainer()
        X_input = trainer._prepare_catboost_data(X_train_raw)
        cat_features = trainer._get_cat_features_indices(X_input)
        model = model_class(**best_params)
        model.fit(
            X_input,
            y_train.values,
            cat_features=cat_features if cat_features else None
        )
    else:
        # All other models — fit on processed data
        model = model_class(**best_params)
        model.fit(X_train, y_train.values)

    tuned_models[model_name] = model
    print(f"Fitted: {model_name}")


Fitting final tuned models on full training data...
Fitted: xgboost_classifier


## 8. Save tuned models and artifacts
All tuned models are saved to `models/tuned/`. The full tuning results
are saved to `artifacts/` for reference and comparison in notebook 7.

In [9]:
TUNED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

for model_name, model in tuned_models.items():
    save_model(model, TUNED_MODELS_DIR / f"{model_name}_tuned.pkl")

print(f"\nTuned models saved to {TUNED_MODELS_DIR}")

Saving model: C:\ML\ML Workflow\Projects\Project Name Template\models\tuned\xgboost_classifier_tuned.pkl


Tuned models saved to C:\ML\ML Workflow\Projects\Project Name Template\models\tuned


In [10]:
joblib.dump(tuned_results, ARTIFACTS_DIR / "tuning_results.pkl")
joblib.dump(tuned_models, ARTIFACTS_DIR / "tuned_models.pkl")
print(f"Tuning artifacts saved to {ARTIFACTS_DIR}")

Tuning artifacts saved to C:\ML\ML Workflow\Projects\Project Name Template\artifacts
